# FLAMES 30 Dor — spatial fit comparison
Click a target to display its existing fit images for maximum 1, 2 and 5 components. Colours identify catalogue samples. The source FITS filename links positions to images; numeric plot prefixes are not used as identifiers.

**Setup:** edit the paths below, then run all cells in order. This requires the catalogue `df-flames-grid.csv` and the existing image folders. No original spectra, GaussPy installation or refitting are needed. Optionally supply `skv_30dor_dss.fits` for the same sky background used in `complete_grid`.

The clickable map uses `ipympl` with `ipywidgets` and a live Jupyter kernel ([documentation](https://matplotlib.org/ipympl/)). If installing packages for the first time, run the next cell after uncommenting it, restart the kernel, then run all cells. Do not switch to `%matplotlib inline`.

Use **Map labels** to switch between decimal degrees and sexagesimal (RA in hours, Dec in degrees), with or without the DSS background. Filter bounds and catalogue coordinates remain in decimal degrees.


In [1]:
# %pip install ipympl ipywidgets matplotlib numpy pandas pillow astropy

In [2]:
%matplotlib widget
from pathlib import Path
from collections import defaultdict
import html
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

DIR = Path(r"C:\Users\javas\Dropbox\GitHub\VIENTO\structure_function_analysis\FLAMES_30dor\gausspyp")
folders = {
    "Maximum 1 component": DIR / "gausspyp_v02.1" / "gausspy_validation" / "plots_validation",
    "Maximum 2 components": DIR / "gausspyp_v02.2" / "gausspy_validation" / "plots_validation",
    "Maximum 5 components": DIR / "gausspyp_v02.3" / "gausspy_validation" / "plots_validation",
}
# Change this if the catalogue is stored elsewhere.
CATALOGUE = DIR / "gausspyp_v02.1" / "df-flames-grid.csv"
# Optional: Path(r"C:\...\skv_30dor_dss.fits"); None gives a coordinate map.
BACKGROUND_FITS = DIR / "skv_30dor_dss.fits"
CLICK_RADIUS_PIXELS = 9


## Match positions to images
The existing pattern `index_object_ADP....fits.png` and the proposed `index_object_RA..._DEC..._ADP....png` are supported. Matching uses a sanitized source filename suffix, never the leading index. Missing images remain visible as missing panels. Multiple matches are reported and selectable rather than chosen silently. `match_report` records availability per source and run.


In [3]:
def safe_string(value):
    return re.sub(r"[^A-Za-z0-9_.-]", "_", str(value))

def source_basename(value):
    return str(value).replace("\\", "/").rsplit("/", 1)[-1]

def match_images(table, run_folders):
    aliases = defaultdict(set)
    for source in table["filename"]:
        name = safe_string(source_basename(source))
        aliases[name].add(source)
        if name.lower().endswith(".fits"):
            aliases[name[:-5]].add(source)
    ambiguous = {k: v for k, v in aliases.items() if len(v) > 1}
    if ambiguous:
        raise ValueError(f"Source names collide after sanitization: {ambiguous}")
    matches, unmatched = {}, {}
    for label, folder in run_folders.items():
        by_source = defaultdict(list)
        unused = []
        if not folder.is_dir():
            print(f"Missing image folder: {folder}")
        else:
            for path in sorted(folder.iterdir()):
                if not path.is_file() or path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                    continue
                # Longest suffix avoids accidental matches to shorter source names.
                hits = [key for key in aliases if path.stem == key or path.stem.endswith("_" + key)]
                if hits:
                    key = max(hits, key=len)
                    by_source[next(iter(aliases[key]))].append(path)
                else:
                    unused.append(path.name)
        matches[label] = by_source
        unmatched[label] = unused
    return matches, unmatched

df = pd.read_csv(CATALOGUE)
required = {"filename", "ra_deg", "dec_deg"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing catalogue columns: {required - set(df.columns)}")
if df["filename"].isna().any() or df["filename"].duplicated().any():
    raise ValueError("Catalogue must have one nonempty, unique filename per observation.")
df["filename"] = df["filename"].astype(str)
for col in ["ra_deg", "dec_deg"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
valid = np.isfinite(df[["ra_deg", "dec_deg"]]).all(axis=1) & df.dec_deg.between(-90, 90) & df.ra_deg.between(0, 360)
print(f"Excluded {int((~valid).sum())} rows with invalid coordinates.")
df = df.loc[valid].reset_index(drop=True)
if df.empty:
    raise ValueError("No valid sky coordinates.")
for col in ["object", "sample"]:
    if col not in df:
        df[col] = "Unclassified" if col == "sample" else ""
    df[col] = df[col].fillna("Unclassified" if col == "sample" else "").astype(str)
matches, unmatched_images = match_images(df, folders)
match_report = df[["filename", "object", "ra_deg", "dec_deg", "sample"]].copy()
for label in folders:
    match_report[label] = [len(matches[label].get(source, [])) for source in df.filename]
    print(f"{label}: {(match_report[label] > 0).sum()} matched; "
          f"{(match_report[label] == 0).sum()} missing; "
          f"{(match_report[label] > 1).sum()} ambiguous; "
          f"{len(unmatched_images[label])} unlinked images")
display(match_report.head())


Excluded 0 rows with invalid coordinates.
Maximum 1 component: 1382 matched; 0 missing; 0 ambiguous; 0 unlinked images
Maximum 2 components: 1382 matched; 0 missing; 0 ambiguous; 0 unlinked images
Maximum 5 components: 1382 matched; 0 missing; 0 ambiguous; 0 unlinked images


,filename,object,ra_deg,dec_deg,sample,Maximum 1 component,Maximum 2 components,Maximum 5 components
0,ADP.2019-01-31T23_53_42.052.fits,pos078,84.550875,-69.148917,irr,1,1,1
1,ADP.2019-01-31T23_53_42.083.fits,pos011,84.634750,-69.088028,irr,1,1,1
2,ADP.2019-01-31T23_53_42.122.fits,pos069,84.852375,-69.082583,irr,1,1,1
3,ADP.2019-01-31T23_53_42.163.fits,pos037,84.610125,-69.163389,irr,1,1,1
4,ADP.2019-01-31T23_53_42.236.fits,pos093,84.460167,-69.170167,irr,1,1,1


## Explore
- Click a point; the selected source is highlighted and its three images appear below.
- If several observations fall within the click radius, choose among them in **Near click**. Zoom in to separate close targets. Turn off the toolbar's pan/zoom mode before selecting.
- **Spectrum** and Previous/Next navigate the filtered catalogue. **Near click** only lists candidates from the most recent click.
- Sample and RA/Dec bounds filter the map and navigation. Bounds are inclusive and sorted automatically. This local 30 Dor filter does not handle RA wrap at 0°.
- **All 3 images** limits selection to sources present in every run. An ambiguous image gets its own file selector.
- A maximum component count is a run setting, not the measured number in each fit.

Without a background, RA increases to the left and the aspect ratio applies the local cos(Dec) correction. With a DSS background, positions use the FITS celestial WCS. Re-running this cell closes the previous viewer.


In [5]:
class SpatialFitViewer:
    def __init__(self, table, image_matches, background=None):
        self.table = table
        self.matches = image_matches
        self.ids = []
        self.busy = False
        self.sample = widgets.Dropdown(options=["All samples"] + sorted(table["sample"].unique()), description="Sample:")
        self.coordinate_format = widgets.ToggleButtons(
            options=["Decimal degrees", "Sexagesimal"], value="Decimal degrees",
            description="Map labels:", style={"description_width": "initial"})
        self.complete = widgets.Checkbox(value=False, description="All 3 images")
        self.bounds = [widgets.FloatText(value=float(v), description=label, layout=widgets.Layout(width="220px"))
                       for label, v in [("RA lower", table.ra_deg.min()), ("RA upper", table.ra_deg.max()),
                                        ("Dec lower", table.dec_deg.min()), ("Dec upper", table.dec_deg.max())]]
        self.apply = widgets.Button(description="Apply filters")
        self.reset = widgets.Button(description="Reset filters")
        self.previous = widgets.Button(description="← Previous")
        self.following = widgets.Button(description="Next →")
        self.selector = widgets.Dropdown(description="Spectrum:", layout=widgets.Layout(width="95%"))
        self.near = widgets.Dropdown(description="Near click:", layout=widgets.Layout(width="95%"))
        self.status = widgets.HTML()
        self.details = widgets.HTML()
        self.panels = widgets.HBox(layout=widgets.Layout(width="100%", overflow="auto"))
        with plt.ioff():
            self.fig = plt.figure(figsize=(9, 6.5), constrained_layout=True)
            if background is None:
                self.ax = self.fig.add_subplot(111)
                self.world_transform = self.ax.transData
                self.ax.set_xlabel("RA (deg)")
                self.ax.set_ylabel("Dec (deg)")
                self.ax.set_aspect(1 / np.cos(np.deg2rad(table.dec_deg.median())))
            else:
                from astropy.io import fits
                from astropy.wcs import WCS
                with fits.open(background) as hdul:
                    data = np.array(hdul[0].data, dtype=float).squeeze()
                    wcs = WCS(hdul[0].header).celestial
                if data.ndim != 2 or not np.isfinite(data).any():
                    raise ValueError("Background must contain a finite 2D image in the primary HDU.")
                self.ax = self.fig.add_subplot(111, projection=wcs)
                lo, hi = np.nanpercentile(data, [1, 99])
                self.ax.imshow(data, origin="lower", cmap="gray_r", vmin=lo, vmax=hi)
                self.world_transform = self.ax.get_transform("world")
                self.ax.coords[0].set_axislabel("RA")
                self.ax.coords[1].set_axislabel("Dec")
            self.background = background
            self.ax.grid(alpha=0.25, linestyle="--")
            self.highlight = self.ax.scatter([], [], s=180, facecolors="none", edgecolors="red",
                                             linewidths=2, transform=self.world_transform, zorder=10)
        if not isinstance(self.fig.canvas, widgets.Widget):
            raise RuntimeError("Run %matplotlib widget before creating the viewer, then rerun this cell.")
        self.artists = []
        self.colors = {s: plt.get_cmap("tab20")(i % 20) for i, s in enumerate(sorted(table["sample"].unique()))}
        self.selector.observe(self.select_changed, names="value")
        self.near.observe(self.near_changed, names="value")
        self.previous.on_click(lambda _: self.navigate(-1))
        self.following.on_click(lambda _: self.navigate(1))
        self.apply.on_click(lambda _: self.filter())
        self.reset.on_click(self.reset_filters)
        self.sample.observe(lambda _: self.filter(), names="value")
        self.complete.observe(lambda _: self.filter(), names="value")
        self.coordinate_format.observe(self.update_coordinate_format, names="value")
        self.update_coordinate_format()
        self.connection = self.fig.canvas.mpl_connect("button_press_event", self.on_click)
        self.ui = widgets.VBox([widgets.HBox([self.sample, self.complete]),
            self.coordinate_format,
            widgets.HTML("Region bounds below are always in decimal degrees."),
            widgets.HBox(self.bounds[:2]), widgets.HBox(self.bounds[2:]),
            widgets.HBox([self.apply, self.reset]), self.status, self.fig.canvas,
            self.near, widgets.HBox([self.previous, self.following]), self.selector, self.details, self.panels])
        self.filter()

    def update_coordinate_format(self, change=None):
        """Change tick labels only, preserving selection, zoom and sky coordinates."""
        sexagesimal = self.coordinate_format.value == "Sexagesimal"
        ra_label = "RA (h:m:s)" if sexagesimal else "RA (deg)"
        dec_label = "Dec (d:m:s)" if sexagesimal else "Dec (deg)"
        if self.background is not None:
            self.ax.coords[0].set_major_formatter("hh:mm:ss.s" if sexagesimal else "d.ddd")
            self.ax.coords[1].set_major_formatter("dd:mm:ss" if sexagesimal else "d.ddd")
            self.ax.coords[0].set_axislabel(ra_label)
            self.ax.coords[1].set_axislabel(dec_label)
        else:
            from matplotlib.ticker import FuncFormatter
            def sex_label(value, is_ra):
                if not np.isfinite(value):
                    return ""
                # Integer tenths of seconds avoid rounding to an invalid 60 seconds.
                angle = (value % 360) / 15 if is_ra else abs(value)
                ticks = int(round(angle * 36000))
                if is_ra:
                    ticks %= 24 * 36000
                major, rest = divmod(ticks, 36000)
                minute, second = divmod(rest, 600)
                sign = "" if is_ra else ("-" if np.signbit(value) else "+")
                return f"{sign}{major:02d}:{minute:02d}:{second/10:04.1f}"
            self.ax.xaxis.set_major_formatter(FuncFormatter(
                lambda x, pos: sex_label(x, True) if sexagesimal else f"{x:.3f}"))
            self.ax.yaxis.set_major_formatter(FuncFormatter(
                lambda y, pos: sex_label(y, False) if sexagesimal else f"{y:.3f}"))
            self.ax.set_xlabel(ra_label)
            self.ax.set_ylabel(dec_label)
        self.fig.canvas.draw_idle()

    def label(self, i):
        row = self.table.loc[i]
        return f"{row['object']} | {row.ra_deg:.5f}, {row.dec_deg:+.5f} | {row.filename}"

    def filter(self):
        if self.busy:
            return
        values = [w.value for w in self.bounds]
        if not np.isfinite(values).all():
            self.status.value = "Enter finite coordinate bounds."
            return
        r0, r1 = sorted(values[:2]); d0, d1 = sorted(values[2:])
        mask = self.table.ra_deg.between(r0, r1) & self.table.dec_deg.between(d0, d1)
        if self.sample.value != "All samples":
            mask &= self.table["sample"].eq(self.sample.value)
        if self.complete.value:
            mask &= self.table.filename.map(lambda f: all(self.matches[l].get(f) for l in self.matches))
        old = self.selector.value
        self.ids = self.table.index[mask].tolist()
        self.busy = True
        self.near.options = []
        self.selector.options = [(self.label(i), i) for i in self.ids]
        self.selector.value = old if old in self.ids else (self.ids[0] if self.ids else None)
        self.busy = False
        for artist in self.artists:
            artist.remove()
        self.artists = []
        for sample, group in self.table.loc[self.ids].groupby("sample"):
            artist = self.ax.scatter(group.ra_deg, group.dec_deg, s=5, color=self.colors[sample],
                                     label=f"{sample} ({len(group)})", transform=self.world_transform)
            self.artists.append(artist)
        legend = self.ax.get_legend()
        if legend is not None:
            legend.remove()
        if self.ids:
            self.ax.legend(fontsize=8, loc="best")
        if self.background is None:
            dx = max((r1-r0)*0.04, 0.001); dy = max((d1-d0)*0.04, 0.001)
            self.ax.set_xlim(r1+dx, r0-dx); self.ax.set_ylim(d0-dy, d1+dy)
        self.ax.set_title("FLAMES 30 Dor — click a target")
        self.status.value = f"<b>{len(self.ids)} / {len(self.table)} observations</b> in the current filter."
        self.show_selected()

    def reset_filters(self, _):
        self.busy = True
        self.sample.value = "All samples"; self.complete.value = False
        for widget, value in zip(self.bounds, [self.table.ra_deg.min(), self.table.ra_deg.max(),
                                               self.table.dec_deg.min(), self.table.dec_deg.max()]):
            widget.value = float(value)
        self.busy = False
        self.filter()

    def select_changed(self, _):
        if not self.busy:
            self.show_selected()

    def near_changed(self, change):
        if not self.busy and change["new"] is not None:
            self.selector.value = change["new"]

    def navigate(self, step):
        if self.selector.value is not None:
            index = self.ids.index(self.selector.value)
            self.selector.value = self.ids[max(0, min(index + step, len(self.ids)-1))]

    def on_click(self, event):
        if event.inaxes is not self.ax or event.button != 1 or not self.ids:
            return
        toolbar = getattr(self.fig.canvas, "toolbar", None)
        if toolbar is not None and getattr(toolbar, "mode", ""):
            return
        coords = self.table.loc[self.ids, ["ra_deg", "dec_deg"]].to_numpy()
        pixels = self.world_transform.transform(coords)
        distance = np.linalg.norm(pixels - [event.x, event.y], axis=1)
        candidates = np.flatnonzero(np.isfinite(distance) & (distance <= CLICK_RADIUS_PIXELS))
        if not len(candidates):
            return
        candidates = candidates[np.argsort(distance[candidates], kind="stable")]
        hits = [self.ids[j] for j in candidates]
        self.busy = True
        self.near.options = [(self.label(i), i) for i in hits]
        self.near.value = hits[0]
        self.busy = False
        self.selector.value = hits[0]

    def image_panel(self, label, paths):
        heading = widgets.HTML(value=f"<b>{html.escape(label)}</b>")
        box_layout = widgets.Layout(width="33%", min_width="300px")
        if not paths:
            return widgets.VBox([heading, widgets.HTML(value="No matching image for this source.")], layout=box_layout)
        picture = widgets.Image(layout=widgets.Layout(width="100%", height="auto"))
        error = widgets.HTML()
        def load(path):
            try:
                # Validate before sending the original bytes to the browser.
                with Image.open(path) as im:
                    im.verify()
                picture.format = "jpeg" if path.suffix.lower() in {".jpg", ".jpeg"} else "png"
                picture.value = path.read_bytes()
                error.value = ""
            except Exception as exc:
                picture.value = b""
                error.value = html.escape(f"Could not load image: {exc}")
        content = [heading]
        if len(paths) > 1:
            choice = widgets.Dropdown(options=[(p.name, str(p)) for p in paths],
                                      description="Image:", layout=widgets.Layout(width="100%"))
            choice.observe(lambda c: load(Path(c["new"])), names="value")
            content += [widgets.HTML(value=f"{len(paths)} matching images — select the intended export."), choice]
        content += [picture, error]
        load(paths[0])
        return widgets.VBox(content, layout=box_layout)

    def show_selected(self):
        i = self.selector.value
        if i is None:
            self.highlight.set_offsets(np.empty((0, 2)))
            self.details.value = "No observations match these filters."
            self.panels.children = ()
            self.previous.disabled = self.following.disabled = True
        else:
            row = self.table.loc[i]
            self.highlight.set_offsets([[row.ra_deg, row.dec_deg]])
            self.details.value = f"<b>{html.escape(row['object'])}</b> | RA {row.ra_deg:.5f}°, Dec {row.dec_deg:+.5f}°<br>{html.escape(row.filename)}"
            self.panels.children = tuple(self.image_panel(label, lookup.get(row.filename, [])) for label, lookup in self.matches.items())
            index = self.ids.index(i)
            self.previous.disabled = index == 0
            self.following.disabled = index == len(self.ids)-1
        self.fig.canvas.draw_idle()

    def close(self):
        self.fig.canvas.mpl_disconnect(self.connection)
        plt.close(self.fig)
        self.ui.close()

if "viewer" in globals():
    viewer.close()
viewer = SpatialFitViewer(df, matches, BACKGROUND_FITS)
display(viewer.ui)


## Troubleshooting and provenance
If the map is static, verify that the backend cell ran with `%matplotlib widget`. After installing `ipympl`, restart the kernel and rerun all cells. The notebook must remain connected to its Python kernel for clicks and buttons to work.

If images are missing, inspect `match_report` and `unmatched_images`, and check the folder paths. For unusual filename conventions, adapt `safe_string` to the exporter; never fall back to matching numeric indices alone. This viewer links images to source observations but cannot verify that the saved fits were generated with the intended run configuration.

The notebook was checked using synthetic catalogue entries and images, including changed numeric prefixes, overlapping coordinates, missing images, duplicate exports and empty filters. Your real image folders, CSV and optional DSS FITS are local to your computer and were not available for an end-to-end test here.
